In [20]:
import pandas as pd

df_games = pd.read_csv('/home/user1/project/fifa_prediction/historical_matches.csv')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)



In [22]:
df_games['date'] = pd.to_datetime(df_games['date'])
df_games = df_games[df_games['date'] <= '2026-06-01']

In [23]:
print(df_games.shape)
df_games.head()


(49294, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [24]:
df_games.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49289,2026-06-01,Norway,Sweden,3.0,1.0,Friendly,Oslo,Norway,False
49290,2026-06-01,Slovakia,Malta,2.0,1.0,Friendly,Dunajská Streda,Slovakia,False
49291,2026-06-01,Turkey,North Macedonia,4.0,0.0,Friendly,Istanbul,Turkey,False
49292,2026-06-01,Maldives,Afghanistan,0.0,1.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False
49307,2026-06-01,Maldives,Pakistan,0.0,3.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False


In [25]:
df_games['tournament'].value_counts().head()


tournament
Friendly                                18285
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            964
Name: count, dtype: int64

In [26]:
df_games = df_games[~df_games['tournament'].str.startswith('CONIFA')]
df_games = df_games[~df_games['tournament'].str.startswith('ConIFA')]

print(df_games.shape)

(49122, 9)


In [28]:
import pandas as pd
from collections import defaultdict

# ============================================================
# Compute Elo ratings for every international team over history
# ============================================================

# ---------- 1. K-factor tiers by tournament importance ----------
TIER_1_WORLD_CUP = {'FIFA World Cup'}

TIER_2_CONTINENTAL = {
    'UEFA Euro', 'Copa América', 'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'CONCACAF Championship', 'Oceania Nations Cup',
    'Confederations Cup',
}

TIER_3_QUALIFIERS_NATIONS = {
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Gold Cup qualification', 'CONCACAF Championship qualification',
    'Copa América qualification', 'Oceania Nations Cup qualification',
    'UEFA Nations League', 'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
}

TIER_4_REGIONAL = {
    # Africa
    'CECAFA Cup', 'COSAFA Cup', 'COSAFA Cup qualification', 'WAFF Championship',
    'Amílcar Cabral Cup', 'All-African Games', 'UDEAC Cup', 'UNIFFAC Cup',
    'West African Cup', 'Nile Basin Tournament', 'African Friendship Games',
    # Asia / Oceania
    'Gulf Cup', 'Arab Cup', 'Arab Cup qualification', 'SAFF Cup',
    'AFF Championship', 'AFF Championship qualification', 'EAFF Championship',
    'EAFF Championship qualification', 'ASEAN Championship',
    'ASEAN Championship qualification', 'AFC Challenge Cup',
    'AFC Challenge Cup qualification', 'Asian Games', 'CAFA Nations Cup',
    'Southeast Asian Games', 'South Asian Games', 'Dynasty Cup',
    'Pacific Games', 'South Pacific Games', 'Melanesia Cup',
    'Indian Ocean Island Games', 'Afro-Asian Games',
    # Europe
    'British Home Championship', 'Nordic Championship', 'Baltic Cup',
    'Balkan Cup', 'Central European International Cup',
    # Americas
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification', 'UNCAF Cup',
    'Central American and Caribbean Games', 'Pan American Championship',
    'CCCF Championship', 'Bolivarian Games', 'NAFC Championship',
    # Multi-sport
    'Olympic Games',
}

TIER_5_FRIENDLY = {'Friendly', 'FIFA Series', 'CONCACAF Series'}


def get_k_factor(tournament):
    """K controls how reactive Elo is to a match. Higher = bigger swings."""
    if tournament in TIER_1_WORLD_CUP:          return 60
    if tournament in TIER_2_CONTINENTAL:        return 50
    if tournament in TIER_3_QUALIFIERS_NATIONS: return 40
    if tournament in TIER_4_REGIONAL:           return 30
    if tournament in TIER_5_FRIENDLY:           return 20
    return 15   # minor/exhibition tournaments, anniversary cups, etc.


# ---------- 2. Expected score formula ----------
def expected_score(home_elo, away_elo, home_advantage=100):
    # home_advantage = 100 → ~64% win rate for evenly matched home teams.
    # Set to 0 for neutral-venue matches (the formula then becomes symmetric).
    diff = (home_elo + home_advantage) - away_elo
    return 1 / (1 + 10 ** (-diff / 400))


# ---------- 3. Load and prepare ----------
df = df_games.copy()
df = df.dropna(subset=['home_score', 'away_score'])

# Drop CONIFA tournaments — these only involve non-FIFA teams
df = df[~df['tournament'].str.contains('CONIFA', case=False, na=False)]

df = df.sort_values('date').reset_index(drop=True)
print(f"Computing Elo over {len(df):,} historical matches")

# ---------- 4. Iterate through history ----------
INITIAL_ELO = 1500
current_elo = defaultdict(lambda: INITIAL_ELO)
elo_history = []

for row in df.itertuples(index=False):
    home, away = row.home_team, row.away_team
    home_elo, away_elo = current_elo[home], current_elo[away]

    # Neutral venue cancels home advantage
    h_adv = 0 if row.neutral else 100
    exp_home = expected_score(home_elo, away_elo, h_adv)
    exp_away = 1 - exp_home

    # Actual result
    if row.home_score > row.away_score:
        actual_home, actual_away = 1.0, 0.0
    elif row.home_score < row.away_score:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # Goal-difference multiplier — bigger wins move Elo more
    goal_diff = abs(row.home_score - row.away_score)
    if goal_diff <= 1:
        g = 1.0
    elif goal_diff == 2:
        g = 1.5
    else:
        g = (11 + goal_diff) / 8

    K = get_k_factor(row.tournament)

    # Update both teams (zero-sum: total Elo across the two teams is preserved)
    current_elo[home] = home_elo + K * g * (actual_home - exp_home)
    current_elo[away] = away_elo + K * g * (actual_away - exp_away)

    elo_history.append((row.date, home, current_elo[home]))
    elo_history.append((row.date, away, current_elo[away]))

# ---------- 5. Build the lookup table ----------
df_elo = pd.DataFrame(elo_history, columns=['date', 'team', 'elo_after'])
df_elo['date'] = pd.to_datetime(df_elo['date'])
print(f"df_elo shape: {df_elo.shape}")

# ---------- 6. Sanity checks ----------
print("\n--- Top 15 current Elo ---")
current_ranking = (
    df_elo.sort_values('date')
          .groupby('team')
          .tail(1)
          .sort_values('elo_after', ascending=False)
          .head(15)
)
print(current_ranking.to_string(index=False))

print("\n--- Bottom 15 current Elo (sanity: should be tiny non-FIFA or minnow teams) ---")
print(
    df_elo.sort_values('date')
          .groupby('team')
          .tail(1)
          .sort_values('elo_after')
          .head(15)
          .to_string(index=False)
)

print(f"\nTotal unique teams with Elo: {df_elo['team'].nunique()}")

Computing Elo over 49,122 historical matches
df_elo shape: (98244, 3)

--- Top 15 current Elo ---
      date        team   elo_after
2026-03-31       Spain 2220.629549
2026-03-31   Argentina 2179.923665
2026-03-29      France 2137.051700
2026-03-31     England 2078.845108
2026-05-31      Brazil 2068.029879
2026-06-01    Colombia 2054.045724
2026-03-31    Portugal 2034.622032
2026-05-30     Ecuador 2020.558657
2026-03-31 Netherlands 2016.453329
2026-05-31     Germany 1990.915054
2026-05-26     Morocco 1982.119812
2026-03-31     Croatia 1982.062579
2026-05-31       Japan 1980.318676
2026-03-31     Uruguay 1971.265813
2026-05-30      Mexico 1968.797271

--- Bottom 15 current Elo (sanity: should be tiny non-FIFA or minnow teams) ---
      date                         team  elo_after
2026-03-29                        Macau 856.061690
2025-11-18                       Bhutan 875.824275
2026-03-29                     Anguilla 898.995083
2025-11-18                       Brunei 906.606543
2026-0

In [29]:
df_games['date'] = pd.to_datetime(df_games['date'])
df_elo['date']   = pd.to_datetime(df_elo['date'])

df_elo_sorted   = df_elo.sort_values('date').reset_index(drop=True)
df_games_sorted = df_games.sort_values('date').reset_index(drop=True)

# ----- HOME: pre-match Elo (lookup strictly BEFORE the match date) -----
df_temp = pd.merge_asof(
    df_games_sorted,
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_pre'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=False,   # exclude same-day entries → no leakage
)

# ----- HOME: post-match Elo (lookup ON the match date — this is the leak) -----
df_temp = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_after'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=True,    # INCLUDE same-day → grabs the post-match update
)

# ----- AWAY: pre-match Elo -----
df_temp = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_pre'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=False,
)

# ----- AWAY: post-match Elo (the leak again) -----
df_match_features = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_after'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=True,
)

df_match_features['elo_diff'] = (
    df_match_features['home_elo_pre'] - df_match_features['away_elo_pre']
)

# Inspect: pre vs after side by side
df_match_features[[
    'date', 'home_team', 'away_team', 'home_score', 'away_score',
    'home_elo_pre', 'home_elo_after', 'away_elo_pre', 'away_elo_after'
]].tail(10)

,date,home_team,away_team,home_score,away_score,home_elo_pre,home_elo_after,away_elo_pre,away_elo_after
49112,2026-05-31,Japan,Iceland,1.0,0.0,1978.942028,1980.318676,1626.448100,1625.071452
49113,2026-06-01,Austria,Tunisia,1.0,0.0,1880.174293,1884.202342,1740.868977,1736.840928
49114,2026-06-01,Bulgaria,Montenegro,0.0,1.0,1533.997421,1520.075102,1490.007975,1503.930293
49115,2026-06-01,Canada,Uzbekistan,2.0,0.0,1889.881666,1898.087992,1820.208991,1812.002665
49116,2026-06-01,Colombia,Costa Rica,3.0,1.0,2051.361719,2054.045724,1748.308191,1745.624186
49117,2026-06-01,Norway,Sweden,3.0,1.0,1961.224460,1965.991642,1771.742825,1766.975644
49118,2026-06-01,Slovakia,Malta,2.0,1.0,1725.366386,1726.437878,1326.515389,1325.443897
49119,2026-06-01,Turkey,North Macedonia,4.0,0.0,1954.227181,1957.502469,1646.591037,1643.315750
49120,2026-06-01,Maldives,Afghanistan,0.0,1.0,1036.288233,1030.674696,1225.594413,1231.207950
49121,2026-06-01,Maldives,Pakistan,0.0,3.0,1036.288233,1030.674696,1030.559852,1047.365556


In [30]:
df_games['date'] = pd.to_datetime(df_games['date'])
df_elo['date'] = pd.to_datetime(df_elo['date'])

df_elo_sorted = df_elo.sort_values('date').reset_index(drop=True)

# Use merge_asof for "find the most recent Elo entry before this match date"
df_games_sorted = df_games.sort_values('date').reset_index(drop=True)

# Home team Elo at time of match
df_with_home_elo = pd.merge_asof(
    df_games_sorted,
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_pre'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=False  # don't pick up the *current* match's post-Elo
)

# Same for away team
df_match_features = pd.merge_asof(
    df_with_home_elo.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_pre'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=False
)

df_match_features['elo_diff'] = df_match_features['home_elo_pre'] - df_match_features['away_elo_pre']


In [31]:
print(df_match_features.shape)
df_match_features.tail()


(49122, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff
49117,2026-06-01,Norway,Sweden,3.0,1.0,Friendly,Oslo,Norway,False,1961.224460,1771.742825,189.481635
49118,2026-06-01,Turkey,North Macedonia,4.0,0.0,Friendly,Istanbul,Turkey,False,1954.227181,1646.591037,307.636144
49119,2026-06-01,Slovakia,Malta,2.0,1.0,Friendly,Dunajská Streda,Slovakia,False,1725.366386,1326.515389,398.850996
49120,2026-06-01,Maldives,Afghanistan,0.0,1.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1225.594413,-189.306181
49121,2026-06-01,Maldives,Pakistan,0.0,3.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1030.559852,5.728380


In [32]:
assert 'home_elo_after' not in df_match_features.columns
assert 'away_elo_after' not in df_match_features.columns
assert 'home_elo_change' not in df_match_features.columns

In [33]:
df_match_features.to_csv('/home/user1/project/fifa_prediction/df_match_features.csv', index=False)
print(f"Saved: {df_match_features.shape}")


Saved: (49122, 12)


In [34]:
df_match_features.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff
49117,2026-06-01,Norway,Sweden,3.0,1.0,Friendly,Oslo,Norway,False,1961.224460,1771.742825,189.481635
49118,2026-06-01,Turkey,North Macedonia,4.0,0.0,Friendly,Istanbul,Turkey,False,1954.227181,1646.591037,307.636144
49119,2026-06-01,Slovakia,Malta,2.0,1.0,Friendly,Dunajská Streda,Slovakia,False,1725.366386,1326.515389,398.850996
49120,2026-06-01,Maldives,Afghanistan,0.0,1.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1225.594413,-189.306181
49121,2026-06-01,Maldives,Pakistan,0.0,3.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1030.559852,5.728380
